# Initial Classification with No Data Preparation

### Loading the Data of Notebook 1

In [1]:
# ══════════════════════════════════════════════════════════════
# NO-NORMALIZATION EXPERIMENTS — DATA LOADING
# Loads chronological 70/10/20 split without any normalization
# Classifiers to run:
#   SVM
#   GRU
#   PatchTST
#   InceptionTime
# ══════════════════════════════════════════════════════════════

import os
import pickle
from collections import Counter

DATA_DIR = "./final_split_data_noNorm"
RESULTS_DIR = "./results_noNorm"
os.makedirs(RESULTS_DIR, exist_ok=True)

def load_bundle(filepath):
    with open(filepath, "rb") as f:
        bundle = pickle.load(f)

    X = bundle["X"]
    y = bundle["y"]

    print(f"Loaded: {filepath}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Class counts: {Counter(y)}")
    print("-" * 55)

    return X, y


X_train, y_train = load_bundle(os.path.join(DATA_DIR, "train_set.pkl"))
X_val,   y_val   = load_bundle(os.path.join(DATA_DIR, "val_set.pkl"))
X_test,  y_test  = load_bundle(os.path.join(DATA_DIR, "test_set.pkl"))

print("No-normalization data loaded successfully.")

Loaded: ./final_split_data_noNorm/train_set.pkl
X shape: (12455, 288, 10)
y shape: (12455,)
Class counts: Counter({np.int64(0): 12337, np.int64(1): 118})
-------------------------------------------------------
Loaded: ./final_split_data_noNorm/val_set.pkl
X shape: (1780, 288, 10)
y shape: (1780,)
Class counts: Counter({np.int64(0): 1763, np.int64(1): 17})
-------------------------------------------------------
Loaded: ./final_split_data_noNorm/test_set.pkl
X shape: (3559, 288, 10)
y shape: (3559,)
Class counts: Counter({np.int64(0): 3525, np.int64(1): 34})
-------------------------------------------------------
No-normalization data loaded successfully.


### Vectorization for SVM

In [13]:
# ══════════════════════════════════════════════════════════════
# FAST FEATURE EXTRACTION FOR SVM — NO CATCH22
# Converts MVTS data from (samples, timesteps, features)
# to simple statistical features per channel.
# Much faster than catch22.
# ══════════════════════════════════════════════════════════════

import numpy as np

def fast_mvts_features(X):
    """
    X shape: (N, T, F)

    Output:
        features per channel:
        mean, std, min, max, median, q25, q75, first, last, range
    Final shape:
        (N, F * 10)
    """

    mean_feat = np.mean(X, axis=1)
    std_feat = np.std(X, axis=1)
    min_feat = np.min(X, axis=1)
    max_feat = np.max(X, axis=1)
    median_feat = np.median(X, axis=1)
    q25_feat = np.percentile(X, 25, axis=1)
    q75_feat = np.percentile(X, 75, axis=1)
    first_feat = X[:, 0, :]
    last_feat = X[:, -1, :]
    range_feat = max_feat - min_feat

    X_features = np.concatenate(
        [
            mean_feat,
            std_feat,
            min_feat,
            max_feat,
            median_feat,
            q25_feat,
            q75_feat,
            first_feat,
            last_feat,
            range_feat
        ],
        axis=1
    )

    return X_features


X_train_svm = fast_mvts_features(X_train)
X_val_svm   = fast_mvts_features(X_val)
X_test_svm  = fast_mvts_features(X_test)

print("SVM feature shapes:")
print("Train:", X_train_svm.shape)
print("Val  :", X_val_svm.shape)
print("Test :", X_test_svm.shape)

SVM feature shapes:
Train: (12455, 100)
Val  : (1780, 100)
Test : (3559, 100)


## Classification

### Helper Functions 

In [3]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [15]:
# ══════════════════════════════════════════════════════════════
# SVM EXPERIMENT — NO NORMALIZATION
# Uses already extracted SVM features
# Step 1: Search best SVM hyperparameters on validation set
# Step 2: Run final experiment on test set using best hyperparameters
# Saves:
#   svm_nonorm.txt
# ══════════════════════════════════════════════════════════════

import os
import time
import copy
import warnings
import numpy as np

from sklearn.svm import SVC, LinearSVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("═" * 70)
print("SVM experiment on NO-NORMALIZATION data")
print("Using already extracted SVM features")
print("═" * 70)

# ---------------------------------------------------------
# Check extracted feature shapes
# ---------------------------------------------------------

print("SVM feature shapes:")
print(f"Train: {X_train_svm.shape}")
print(f"Val  : {X_val_svm.shape}")
print(f"Test : {X_test_svm.shape}")

d_nonorm = {
    "X_train": X_train_svm,
    "y_train": y_train,
    "X_val": X_val_svm,
    "y_val": y_val,
    "X_test": X_test_svm,
    "y_test": y_test,
}

# ══════════════════════════════════════════════════════════════
# SVM HYPERPARAMETER GRID
# Tune on no-normalization validation set
# ══════════════════════════════════════════════════════════════

SVM_GRID = [
    {"model": LinearSVC, "params": {"C": 0.01, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.05, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.1,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.5,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 1.0,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 3.0,  "class_weight": "balanced", "max_iter": 10000}},

    {"model": SVC, "params": {"kernel": "rbf", "C": 0.01, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 1.0,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
]


def combo_label(combo):
    ModelClass = combo["model"]
    params = combo["params"]

    cw = "balanced" if params.get("class_weight") == "balanced" else "none"

    if ModelClass == LinearSVC:
        return f"LinearSVC | C={params['C']} | cw={cw}"
    else:
        return f"RBF-SVC   | C={params['C']} | gamma={params['gamma']} | cw={cw}"


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON VALIDATION SET
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("SVM hyperparameter search on NO-NORMALIZATION validation set")
print("═" * 70)

best_tss = -np.inf
best_model_class = None
best_params = None
best_val_metrics = None

for i, combo in enumerate(SVM_GRID):

    ModelClass = combo["model"]
    params = copy.deepcopy(combo["params"])

    print(f"[{i + 1}/{len(SVM_GRID)}] {combo_label(combo)}", flush=True)

    model = ModelClass(**params)

    t0 = time.time()
    model.fit(d_nonorm["X_train"], d_nonorm["y_train"])
    y_val_pred = model.predict(d_nonorm["X_val"])
    elapsed = time.time() - t0

    val_metrics = compute_metrics(d_nonorm["y_val"], y_val_pred)

    print(
        f"    Val TSS={val_metrics['tss']:.4f} | "
        f"F1={val_metrics['f1']:.4f} | "
        f"Recall={val_metrics['recall']:.4f} | "
        f"Precision={val_metrics.get('precision', 0):.4f} | "
        f"Time={elapsed:.1f}s\n"
    )

    if val_metrics["tss"] > best_tss:
        best_tss = val_metrics["tss"]
        best_model_class = ModelClass
        best_params = params
        best_val_metrics = val_metrics


print("Best SVM hyperparameters selected from validation set:")
print(f"  Model         : {best_model_class.__name__}")
print(f"  Params        : {best_params}")
print(f"  Val TSS       : {best_tss:.4f}")
print(f"  Val F1        : {best_val_metrics['f1']:.4f}")
print(f"  Val Recall    : {best_val_metrics['recall']:.4f}")
print(f"  Val Precision : {best_val_metrics.get('precision', 0):.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 2: RUN FINAL TEST EXPERIMENT
# Uses best hyperparameters selected from validation
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("Final SVM experiment on NO-NORMALIZATION test set")
print("Using best hyperparameters selected from validation")
print("═" * 70)

print(f"Dataset : No normalization")
print(f"Model   : {best_model_class.__name__}")
print(f"Params  : {best_params}")

metrics_list = []

for run in range(2):

    print(f"\nRun {run + 1}...", flush=True)

    model = best_model_class(**copy.deepcopy(best_params))

    metrics = train_and_evaluate(
        model,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_test"],  d_nonorm["y_test"]
    )

    metrics_list.append(metrics)

    print(
        f"Run {run + 1}: "
        f"TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.2f}s"
    )


# ══════════════════════════════════════════════════════════════
# STEP 3: SAVE RESULTS
# ══════════════════════════════════════════════════════════════

filepath = os.path.join(RESULTS_DIR, "svm_nonorm.txt")

save_results(metrics_list, filepath)
print_results(metrics_list, "SVM | No normalization")


# ══════════════════════════════════════════════════════════════
# STEP 4: SUMMARY
# ══════════════════════════════════════════════════════════════

avg_tss = np.mean([m["tss"] for m in metrics_list])
avg_f1 = np.mean([m["f1"] for m in metrics_list])
avg_recall = np.mean([m["recall"] for m in metrics_list])
avg_precision = np.mean([m.get("precision", 0) for m in metrics_list])

print("\n" + "═" * 70)
print("SVM no-normalization summary")
print("═" * 70)
print(f"Best model      : {best_model_class.__name__}")
print(f"Best params     : {best_params}")
print(f"Validation TSS  : {best_tss:.4f}")
print(f"Test Avg TSS    : {avg_tss:.4f}")
print(f"Test Avg F1     : {avg_f1:.4f}")
print(f"Test Avg Recall : {avg_recall:.4f}")
print(f"Test Avg Prec.  : {avg_precision:.4f}")

print("\nNo-normalization SVM experiment complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print("Saved file: svm_nonorm.txt")

══════════════════════════════════════════════════════════════════════
SVM experiment on NO-NORMALIZATION data
Using already extracted SVM features
══════════════════════════════════════════════════════════════════════
SVM feature shapes:
Train: (12455, 100)
Val  : (1780, 100)
Test : (3559, 100)

══════════════════════════════════════════════════════════════════════
SVM hyperparameter search on NO-NORMALIZATION validation set
══════════════════════════════════════════════════════════════════════
[1/10] LinearSVC | C=0.01 | cw=balanced
    Val TSS=0.2221 | F1=0.0346 | Recall=0.4706 | Precision=0.0000 | Time=272.1s

[2/10] LinearSVC | C=0.05 | cw=balanced
    Val TSS=0.2227 | F1=0.0346 | Recall=0.4706 | Precision=0.0000 | Time=283.0s

[3/10] LinearSVC | C=0.1 | cw=balanced
    Val TSS=0.1633 | F1=0.0303 | Recall=0.4118 | Precision=0.0000 | Time=278.0s

[4/10] LinearSVC | C=0.5 | cw=balanced
    Val TSS=0.1645 | F1=0.0304 | Recall=0.4118 | Precision=0.0000 | Time=275.8s

[5/10] LinearSVC 

### GRU 

In [21]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


### Run GRU Experiments

In [22]:
# ══════════════════════════════════════════════════════════════
# GRU EXPERIMENT — NO NORMALIZATION
# Step 1: Search best GRU hyperparameters on validation set
# Step 2: Run final experiment on test set using best hyperparameters
# Saves:
#   gru_nonorm.txt
# ══════════════════════════════════════════════════════════════

import os
import time
import copy
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("═" * 70)
print("GRU experiment on NO-NORMALIZATION data")
print("Using raw MVTS sequences")
print("═" * 70)

# ---------------------------------------------------------
# Build no-normalization GRU dataset
# Assumes these are already loaded:
#   X_train, y_train, X_val, y_val, X_test, y_test
# Shape should be:
#   X_train: (samples, 288, 10)
# ---------------------------------------------------------

d_nonorm = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
}

print("GRU data shapes:")
print(f"Train: {d_nonorm['X_train'].shape}")
print(f"Val  : {d_nonorm['X_val'].shape}")
print(f"Test : {d_nonorm['X_test'].shape}")

class_ratio = int(
    (d_nonorm["y_train"] == 0).sum() /
    (d_nonorm["y_train"] == 1).sum()
)

print(f"\nTrain class ratio 0:1 = {class_ratio}:1")

# ══════════════════════════════════════════════════════════════
# GRU HYPERPARAMETER GRID
# Tune on no-normalization validation set
# ══════════════════════════════════════════════════════════════

GRU_GRID = [
    {"units": 64,  "layers": 1, "dropout": 0.2, "lr": 0.001,  "batch_size": 32},
    {"units": 64,  "layers": 1, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "layers": 1, "dropout": 0.2, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "layers": 1, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},

    {"units": 64,  "layers": 2, "dropout": 0.2, "lr": 0.001,  "batch_size": 32},
    {"units": 64,  "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "layers": 2, "dropout": 0.2, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "layers": 2, "dropout": 0.3, "lr": 0.001,  "batch_size": 32},

    {"units": 128, "layers": 1, "dropout": 0.3, "lr": 0.0005, "batch_size": 32},
    {"units": 128, "layers": 1, "dropout": 0.3, "lr": 0.001,  "batch_size": 64},
]


def gru_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON VALIDATION SET
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("GRU hyperparameter search on NO-NORMALIZATION validation set")
print("═" * 70)

best_params = None
best_tss = -np.inf
best_val_metrics = None

for i, params in enumerate(GRU_GRID):

    print(f"[{i + 1}/{len(GRU_GRID)}] {gru_label(params)}", flush=True)

    metrics, _ = train_and_evaluate_gru(
        params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        class_ratio=class_ratio
    )

    print(
        f"    Val TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics


print("Best GRU hyperparameters selected from validation set:")
print(f"  Params        : {gru_label(best_params)}")
print(f"  Val TSS       : {best_tss:.4f}")
print(f"  Val F1        : {best_val_metrics['f1']:.4f}")
print(f"  Val Recall    : {best_val_metrics['recall']:.4f}")
print(f"  Val Precision : {best_val_metrics.get('precision', 0):.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 2: RUN FINAL TEST EXPERIMENT
# Uses best hyperparameters selected from validation
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("Final GRU experiment on NO-NORMALIZATION test set")
print("Using best hyperparameters selected from validation")
print("═" * 70)

print(f"Dataset : No normalization")
print(f"Params  : {gru_label(best_params)}")
print(f"Ratio   : {class_ratio}:1")

metrics_list = []

for run in range(2):

    print(f"\nRun {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_gru(
        best_params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_test"],  d_nonorm["y_test"],
        class_ratio=class_ratio
    )

    metrics_list.append(metrics)

    print(
        f"Run {run + 1}: "
        f"TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s"
    )


# ══════════════════════════════════════════════════════════════
# STEP 3: SAVE RESULTS
# ══════════════════════════════════════════════════════════════

filepath = os.path.join(RESULTS_DIR, "gru_nonorm.txt")

save_results(metrics_list, filepath)
print_results(metrics_list, "GRU | No normalization")


# ══════════════════════════════════════════════════════════════
# STEP 4: SUMMARY
# ══════════════════════════════════════════════════════════════

avg_tss = np.mean([m["tss"] for m in metrics_list])
avg_f1 = np.mean([m["f1"] for m in metrics_list])
avg_recall = np.mean([m["recall"] for m in metrics_list])
avg_precision = np.mean([m.get("precision", 0) for m in metrics_list])

print("\n" + "═" * 70)
print("GRU no-normalization summary")
print("═" * 70)
print(f"Best params     : {gru_label(best_params)}")
print(f"Validation TSS  : {best_tss:.4f}")
print(f"Test Avg TSS    : {avg_tss:.4f}")
print(f"Test Avg F1     : {avg_f1:.4f}")
print(f"Test Avg Recall : {avg_recall:.4f}")
print(f"Test Avg Prec.  : {avg_precision:.4f}")

print("\nNo-normalization GRU experiment complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print("Saved file: gru_nonorm.txt")

══════════════════════════════════════════════════════════════════════
GRU experiment on NO-NORMALIZATION data
Using raw MVTS sequences
══════════════════════════════════════════════════════════════════════
GRU data shapes:
Train: (12455, 288, 10)
Val  : (1780, 288, 10)
Test : (3559, 288, 10)

Train class ratio 0:1 = 104:1

══════════════════════════════════════════════════════════════════════
GRU hyperparameter search on NO-NORMALIZATION validation set
══════════════════════════════════════════════════════════════════════
[1/10] units=64  layers=1  drop=0.2  lr=0.001  bs=32
    Val TSS=0.1460 | F1=0.0236 | Recall=0.7059 | Precision=0.0000 | Train=229.3s

[2/10] units=64  layers=1  drop=0.3  lr=0.001  bs=32
    Val TSS=0.1869 | F1=0.0238 | Recall=0.8824 | Precision=0.0000 | Train=316.6s

[3/10] units=128  layers=1  drop=0.2  lr=0.001  bs=32
    Val TSS=0.2238 | F1=0.0251 | Recall=0.8824 | Precision=0.0000 | Train=250.8s

[4/10] units=128  layers=1  drop=0.3  lr=0.001  bs=32
    Val TSS

### PatchTST

In [27]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=20,
    patience=1
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [30]:
# ══════════════════════════════════════════════════════════════
# PATCHTST EXPERIMENT — NO NORMALIZATION
# Step 1: Search best PatchTST hyperparameters on validation set
# Step 2: Run final experiment on test set using best hyperparameters
# Saves:
#   patchtst_nonorm.txt
# ══════════════════════════════════════════════════════════════

import os
import copy
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"{'═' * 70}")
print("  Classifier : Hugging Face PatchTST")
print("  Dataset    : No normalization")
print(f"{'═' * 70}")

# ---------------------------------------------------------
# Build no-normalization dataset dictionary
# Assumes these are already loaded:
#   X_train, y_train, X_val, y_val, X_test, y_test
# Shape should be:
#   X_train: (samples, 288, 10)
# ---------------------------------------------------------

d_nonorm = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
}

timesteps = d_nonorm["X_train"].shape[1]

class_ratio = (
    (d_nonorm["y_train"] == 0).sum() /
    (d_nonorm["y_train"] == 1).sum()
)

print("PatchTST data shapes:")
print(f"Train: {d_nonorm['X_train'].shape}")
print(f"Val  : {d_nonorm['X_val'].shape}")
print(f"Test : {d_nonorm['X_test'].shape}")
print(f"\nTimesteps: {timesteps}")
print(f"Train class ratio 0:1 = {class_ratio:.1f}:1")


# ══════════════════════════════════════════════════════════════
# PATCHTST HYPERPARAMETER GRID
# Tune on no-normalization validation set
# ══════════════════════════════════════════════════════════════

PATCHTST_GRID = [
    # Patch=12 region: better F1 in your earlier run
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 16,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 64,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 32,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 64,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },

    # Coarser patch baseline
    {
        "patch_length": 24,
        "patch_stride": 24,
        "d_model": 64,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
]


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"ffn={p['ffn_dim']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON VALIDATION SET
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("PatchTST hyperparameter search on NO-NORMALIZATION validation set")
print("═" * 70)

best_params = None
best_tss = -np.inf
best_val_metrics = None

for i, params in enumerate(PATCHTST_GRID):

    print(f"[{i + 1}/{len(PATCHTST_GRID)}] {patchtst_label(params)}", flush=True)

    metrics, _ = train_and_evaluate_patchtst(
        params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        class_ratio=class_ratio,
        max_epochs=20,
        patience=1
    )

    print(
        f"    Val TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics


print("Best PatchTST hyperparameters selected from validation set:")
print(f"  Params        : {patchtst_label(best_params)}")
print(f"  Val TSS       : {best_tss:.4f}")
print(f"  Val F1        : {best_val_metrics['f1']:.4f}")
print(f"  Val Recall    : {best_val_metrics['recall']:.4f}")
print(f"  Val Precision : {best_val_metrics.get('precision', 0):.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 2: RUN FINAL TEST EXPERIMENT
# Uses best hyperparameters selected from validation
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("Final PatchTST experiment on NO-NORMALIZATION test set")
print("Using best hyperparameters selected from validation")
print("═" * 70)

print(f"Dataset : No normalization")
print(f"Params  : {patchtst_label(best_params)}")
print(f"Ratio   : {class_ratio:.1f}:1")

metrics_list = []

for run in range(2):

    print(f"\nRun {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_patchtst(
        best_params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_test"],  d_nonorm["y_test"],
        class_ratio=class_ratio,
        max_epochs=20,
        patience=1
    )

    metrics_list.append(metrics)

    print(
        f"Run {run + 1}: "
        f"TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s"
    )


# ══════════════════════════════════════════════════════════════
# STEP 3: SAVE RESULTS
# ══════════════════════════════════════════════════════════════

filepath = os.path.join(RESULTS_DIR, "patchtst_nonorm.txt")

save_results(metrics_list, filepath)
print_results(metrics_list, "PatchTST | No normalization")


# ══════════════════════════════════════════════════════════════
# STEP 4: SUMMARY
# ══════════════════════════════════════════════════════════════

avg_tss = np.mean([m["tss"] for m in metrics_list])
avg_f1 = np.mean([m["f1"] for m in metrics_list])
avg_recall = np.mean([m["recall"] for m in metrics_list])
avg_precision = np.mean([m.get("precision", 0) for m in metrics_list])

print("\n" + "═" * 70)
print("PatchTST no-normalization summary")
print("═" * 70)
print(f"Best params     : {patchtst_label(best_params)}")
print(f"Validation TSS  : {best_tss:.4f}")
print(f"Test Avg TSS    : {avg_tss:.4f}")
print(f"Test Avg F1     : {avg_f1:.4f}")
print(f"Test Avg Recall : {avg_recall:.4f}")
print(f"Test Avg Prec.  : {avg_precision:.4f}")

print("\nNo-normalization PatchTST experiment complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print("Saved file: patchtst_nonorm.txt")

══════════════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
  Dataset    : No normalization
══════════════════════════════════════════════════════════════════════
PatchTST data shapes:
Train: (12455, 288, 10)
Val  : (1780, 288, 10)
Test : (3559, 288, 10)

Timesteps: 288
Train class ratio 0:1 = 104.6:1

══════════════════════════════════════════════════════════════════════
PatchTST hyperparameter search on NO-NORMALIZATION validation set
══════════════════════════════════════════════════════════════════════
[1/4] patch=12  stride=12  d=16  heads=4  layers=1  ffn=64  drop=0.2  lr=0.001  bs=128
    Val TSS=0.0000 | F1=0.0000 | Recall=0.0000 | Precision=0.0000 | Train=30.8s

[2/4] patch=12  stride=12  d=32  heads=4  layers=1  ffn=128  drop=0.2  lr=0.001  bs=128
    Val TSS=0.4256 | F1=0.0861 | Recall=0.5294 | Precision=0.0000 | Train=110.9s

[3/4] patch=12  stride=12  d=64  heads=4  layers=1  ffn=256  drop=0.2  lr=0.001  bs=128
    Val TSS=0.

### InceptionTime

In [4]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME MODEL
# ══════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32
    ):
        super(InceptionModule, self).__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # Input shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ══════════════════════════════════════════════════════════════
# TRAIN AND EVALUATE INCEPTIONTIME
# ══════════════════════════════════════════════════════════════

def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [5]:
# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME EXPERIMENT — NO NORMALIZATION
# Step 1: Search best InceptionTime hyperparameters on validation set
# Step 2: Run final experiment on test set using best hyperparameters
# Saves:
#   inceptiontime_nonorm.txt
# ══════════════════════════════════════════════════════════════

import os
import copy
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"{'═' * 70}")
print("  Classifier : InceptionTime")
print("  Dataset    : No normalization")
print(f"{'═' * 70}")

# ---------------------------------------------------------
# Build no-normalization dataset dictionary
# Assumes these are already loaded:
#   X_train, y_train, X_val, y_val, X_test, y_test
# Shape should be:
#   X_train: (samples, 288, 10)
# ---------------------------------------------------------

d_nonorm = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
}

timesteps = d_nonorm["X_train"].shape[1]

class_ratio = (
    (d_nonorm["y_train"] == 0).sum() /
    (d_nonorm["y_train"] == 1).sum()
)

print("InceptionTime data shapes:")
print(f"Train: {d_nonorm['X_train'].shape}")
print(f"Val  : {d_nonorm['X_val'].shape}")
print(f"Test : {d_nonorm['X_test'].shape}")
print(f"\nTimesteps: {timesteps}")
print(f"Train class ratio 0:1 = {class_ratio:.1f}:1")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME HYPERPARAMETER GRID
# Tune on no-normalization validation set
# ══════════════════════════════════════════════════════════════

INCEPTION_GRID = [
    {"out_channels": 4,  "depth": 2, "dropout": 0.1, "lr": 0.005, "batch_size": 128},
    {"out_channels": 4,  "depth": 2, "dropout": 0.2, "lr": 0.005, "batch_size": 128},
    {"out_channels": 8,  "depth": 4, "dropout": 0.1, "lr": 1e-3, "batch_size": 128},
    {"out_channels": 8,  "depth": 4, "dropout": 0.2, "lr": 1e-3, "batch_size": 128},
    {"out_channels": 16, "depth": 4, "dropout": 0.1, "lr": 1e-3, "batch_size": 64},
    {"out_channels": 16, "depth": 4, "dropout": 0.2, "lr": 1e-3, "batch_size": 128},
    {"out_channels": 16, "depth": 6, "dropout": 0.1, "lr": 1e-3, "batch_size": 128},
    {"out_channels": 16, "depth": 6, "dropout": 0.2, "lr": 1e-3, "batch_size": 64},
    {"out_channels": 16, "depth": 6, "dropout": 0.3, "lr": 1e-3, "batch_size": 128},
    {"out_channels": 32, "depth": 6, "dropout": 0.2, "lr": 1e-3, "batch_size": 64},
]


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON VALIDATION SET
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("InceptionTime hyperparameter search on NO-NORMALIZATION validation set")
print("═" * 70)

best_params = None
best_tss = -np.inf
best_val_metrics = None

for i, params in enumerate(INCEPTION_GRID):

    print(f"[{i + 1}/{len(INCEPTION_GRID)}] {inception_label(params)}", flush=True)

    metrics, _ = train_and_evaluate_inception(
        params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        class_ratio=class_ratio,
        max_epochs=30,
        patience=3
    )

    print(
        f"    Val TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics


print("Best InceptionTime hyperparameters selected from validation set:")
print(f"  Params        : {inception_label(best_params)}")
print(f"  Val TSS       : {best_tss:.4f}")
print(f"  Val F1        : {best_val_metrics['f1']:.4f}")
print(f"  Val Recall    : {best_val_metrics['recall']:.4f}")
print(f"  Val Precision : {best_val_metrics.get('precision', 0):.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 2: RUN FINAL TEST EXPERIMENT
# Uses best hyperparameters selected from validation
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("Final InceptionTime experiment on NO-NORMALIZATION test set")
print("Using best hyperparameters selected from validation")
print("═" * 70)

print(f"Dataset : No normalization")
print(f"Params  : {inception_label(best_params)}")
print(f"Ratio   : {class_ratio:.1f}:1")

metrics_list = []

for run in range(2):

    print(f"\nRun {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_inception(
        best_params,
        d_nonorm["X_train"], d_nonorm["y_train"],
        d_nonorm["X_val"],   d_nonorm["y_val"],
        d_nonorm["X_test"],  d_nonorm["y_test"],
        class_ratio=class_ratio,
        max_epochs=50,
        patience=3
    )

    metrics_list.append(metrics)

    print(
        f"Run {run + 1}: "
        f"TSS={metrics['tss']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"Recall={metrics['recall']:.4f} | "
        f"Precision={metrics.get('precision', 0):.4f} | "
        f"Train={metrics['train_time']:.1f}s"
    )


# ══════════════════════════════════════════════════════════════
# STEP 3: SAVE RESULTS
# ══════════════════════════════════════════════════════════════

filepath = os.path.join(RESULTS_DIR, "inceptiontime_nonorm.txt")

save_results(metrics_list, filepath)
print_results(metrics_list, "InceptionTime | No normalization")


# ══════════════════════════════════════════════════════════════
# STEP 4: SUMMARY
# ══════════════════════════════════════════════════════════════

avg_tss = np.mean([m["tss"] for m in metrics_list])
avg_f1 = np.mean([m["f1"] for m in metrics_list])
avg_recall = np.mean([m["recall"] for m in metrics_list])
avg_precision = np.mean([m.get("precision", 0) for m in metrics_list])

print("\n" + "═" * 70)
print("InceptionTime no-normalization summary")
print("═" * 70)
print(f"Best params     : {inception_label(best_params)}")
print(f"Validation TSS  : {best_tss:.4f}")
print(f"Test Avg TSS    : {avg_tss:.4f}")
print(f"Test Avg F1     : {avg_f1:.4f}")
print(f"Test Avg Recall : {avg_recall:.4f}")
print(f"Test Avg Prec.  : {avg_precision:.4f}")

print("\nNo-normalization InceptionTime experiment complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print("Saved file: inceptiontime_nonorm.txt")

══════════════════════════════════════════════════════════════════════
  Classifier : InceptionTime
  Dataset    : No normalization
══════════════════════════════════════════════════════════════════════
InceptionTime data shapes:
Train: (12455, 288, 10)
Val  : (1780, 288, 10)
Test : (3559, 288, 10)

Timesteps: 288
Train class ratio 0:1 = 104.6:1

══════════════════════════════════════════════════════════════════════
InceptionTime hyperparameter search on NO-NORMALIZATION validation set
══════════════════════════════════════════════════════════════════════
[1/10] channels=4  depth=2  drop=0.1  lr=0.005  bs=128
    Val TSS=-0.0391 | F1=0.0000 | Recall=0.0000 | Precision=0.0000 | Train=40.8s

[2/10] channels=4  depth=2  drop=0.2  lr=0.005  bs=128
    Val TSS=0.0000 | F1=0.0000 | Recall=0.0000 | Precision=0.0000 | Train=42.9s

[3/10] channels=8  depth=4  drop=0.1  lr=0.001  bs=128
    Val TSS=0.0497 | F1=0.0588 | Recall=0.0588 | Precision=0.0000 | Train=113.0s

[4/10] channels=8  depth=4  